# Does scorer accuracy hold as candidates get further from the parent?

`03_adapt_calibration.ipynb` calibrates the scorers against ADAPT, where every
measured candidate carries one, two or three substitutions. Track A designs
carry five and more. So the calibration we have covers a neighbourhood the
designs have already left, and a scorer that ranks singles well is not thereby
shown to rank a six-substitution CDR-H3 well.

This notebook asks that one question against the one block in the evidence set
big enough to answer it: **Chinery 2026**, a Carterra LSA panel of trastuzumab
CDR-H3 redesigns measured as IgG, 156 of which returned a numeric KD. The
substitution counts run 1 to 10, which is the range Track A actually occupies.

**The deliverable is one curve: Spearman against substitution count.** Not the
pooled number. A pooled correlation over a set whose candidates differ in size
mostly measures the size effect, so the size effect is reported here on its own
line and cannot hide inside a headline.

| Part | What it does | Reads | Writes |
|---|---|---|---|
| 0 | Preflight: which multi-mutation scorers have a route on this box | `routes.py` | nothing |
| 1 | The block: 156 designs with a numeric KD, binned by substitution count | the evidence CSV | nothing |
| 2 | The numbering, checked twice, before a model is loaded | the PDB | nothing |
| 3 | One `variant_id` per design, as a variant table | the PDB | `OUT_DIR/variants` |
| 4 | Score: `antifold_seq`, then `foldx`, then `gearbind` if its image is wired in | the table | `OUT_DIR/scores` |
| 5 | Spearman within each substitution count — **the curve** | the scored table | the curve, the table |
| 6 | The confounds, out loud: pooled rho, size-alone rho, the selection | Part 5 | nothing |
| 7 | ADAPT, kept apart | the evidence CSV | nothing |

Parts 0 to 3 and 5 to 7 are cheap and need no GPU. Part 4 is the expensive one
and the only part that needs Env B, a FoldX licence or a container.

## What this notebook adds to Part 6 of `03_adapt_calibration.ipynb`

That notebook already ran `antifold_seq` over this same 156 and already printed
a within-size table, so the `antifold_seq` column here is not new. Three things
are:

* **Two more scorers.** `foldx` and `gearbind` are the other two in the
  registry that take a candidate's substitutions together. One curve from one
  scorer is an anecdote about AntiFold; three curves are a statement about the
  approach.
* **The curve as the output.** A plot rather than a column of numbers, because
  the shape is the finding and the shape is what gets argued about.
* **A regression check.** The `antifold_seq` figures that notebook recorded on
  2026-10-31 are pinned in Part 5 and compared. If they have moved, the
  pipeline moved, and that is worth an error rather than a shrug.

## What this block is, and what it is not

Read this before the numbers, because three properties of the block decide what
the curve is allowed to mean.

**The 156 are the designs that produced a clean 1:1 fit out of 720 submitted.**
That is a selection on outcome. The rest are 445 `N/A`, 102 `I.C.` (binding
seen, not reliably fit) and 14 blank, and the evidence CSV is explicit that none
of those is converted into a demonstrated non-binder — so they cannot be ranked
as "worst" and they are not missing at random either. Both the weakest and the
unmeasurable candidates are gone, the KD range is truncated, and **every
correlation in this notebook is attenuated by that.** Each one is a lower bound.

The truncation is also not uniform across substitution count, which is the part
that actually threatens the curve: a bigger design is likelier to fail to fit at
all, so the surviving large-size bins are the more heavily filtered ones. A
curve that falls with size is therefore consistent with two stories — scorers
degrade with distance, or the large-size bins have had their informative tails
cut off — and **nothing in this notebook can separate them.** Part 6 reports
what it can: the fit rate per substitution count, so the size-dependence of the
selection is at least on the page next to the curve it could be causing.

**It is IgG captured through Fc on Carterra.** Two Fab arms on a
surface-captured antibody against a surface-presented antigen; the 1:1 fit the
authors applied is a model, not evidence that the interaction was monovalent.
ADAPT is Fab, monovalent, SPR, with batch-matched parent normalisation. **The
two are not pooled here in either direction** — separate correlations, separate
tables, Part 7 kept to itself. A design-dependent avidity term is exactly the
sort of thing that would survive pooling as a spurious signal.

**There is no batch-matched wild-type fold.** The block's own `outcome` is
`measurable binding; no matched WT fold assigned`, so the honest ground truth is
the raw KD, not a fold-improvement over a parent measured alongside it. Raw KD
is what is correlated below.

In [ ]:
# --- parameters -------------------------------------------------------------

EVIDENCE = "data/benchmarks/trastuzumab_mutation_evidence.csv"

# The block. `format` matters: the same study's rows are filtered to the IgG
# panel, which is the whole reason this cannot be pooled with ADAPT's Fab.
CHINERY_STUDY = "chinery2026"
CHINERY_FORMAT = "IgG"
CHINERY_EXPECTED = 156
CHINERY_EXPECTED_SIZES = {1: 14, 2: 13, 3: 17, 4: 32, 5: 27, 6: 23, 7: 14, 8: 8, 9: 4, 10: 4}

ADAPT_STUDY = "adapt2017"          # Part 7 only, and never joined to the above

# The local 1N8Z: trastuzumab Fab bound to HER2 domain IV. Its heavy chain is
# 'B', not 'H'; the benchmark writes H and L, so CHAIN_MAP is the one place
# that is translated and Part 2 asserts it rather than trusting it.
PDB_PATH = None                    # None -> the candidates searched in Part 2
HEAVY_CHAIN = "B"
LIGHT_CHAIN = "A"
ANTIGEN_CHAIN = "C"
CHAIN_MAP = {"H": HEAVY_CHAIN, "L": LIGHT_CHAIN}

OUT_DIR = "runs/chinery_size_calibration"

# The scorers, cheapest first. Every one of these takes a candidate's
# substitutions together, which is the entire requirement here: a scorer that
# can only answer a single cannot be asked this question. The plain `antifold`
# matrix scorer is deliberately absent - see Part 4.
MODELS = ("antifold_seq", "foldx", "gearbind")

# Env B's interpreter, the one that can `import antifold`. None falls back to
# $DESIGN_PIPELINE_ANTIFOLD, which is what the adapter reads anyway.
ANTIFOLD_PYTHON = None
ANTIFOLD_WEIGHTS = None            # a path, to reuse the 567 MB download
ANTIFOLD_DEVICE = "cpu"

# FoldX comes out of the environment: FOLDX_BINARY, and DESIGN_PIPELINE_FOLDX_PDB
# pointing at the repaired, trimmed Fv + antigen domain the alanine scan ran on
# (its residue numbers must still be 1N8Z's). FOLDX_GROUPS defaults to AB,C,
# which is already antibody-against-antigen for this structure. None here leaves
# the environment's own value alone.
FOLDX_PYTHON = None                # None -> $DESIGN_PIPELINE_FOLDX
FOLDX_WORKERS = None
FOLDX_RUNS = None

# GearBind has no published image; the registry carries tool=None until you set
# one. Left None, Part 4 reports it unavailable and the notebook carries on with
# the columns it does have.
GEARBIND_IMAGE = None

# score_variants wants a cap for a per-variant scorer. 156 is under
# PER_VARIANT_LIMIT, so this is only here to make the number explicit.
MAX_VARIANTS = 200

MIN_BIN = 10                       # bins smaller than this are reported, but drawn hollow
CACHE = True
SCORE_ADAPT = True                 # Part 7: the same scorers on the ADAPT block

# What Part 6 of 03_adapt_calibration.ipynb recorded on 2026-10-31, per
# substitution count, as Spearman of score_antifold_seq against -KD. Pinned so a
# re-run that disagrees is an error and not a thing somebody notices later.
KNOWN_ANTIFOLD_SEQ = {
    1: +0.415, 2: +0.346, 3: +0.145, 4: -0.011, 5: -0.277,
    6: -0.275, 7: -0.277, 8: -0.405, 9: +0.000, 10: -0.400,
}
KNOWN_ANTIFOLD_SEQ_POOLED = +0.237
KNOWN_COUNT_ALONE = +0.461
KNOWN_TOLERANCE = 0.02             # the recorded figures are printed to 3 decimals

## Part 0 — which of the three can run here

`routes.py` is the table of where each scorer lives, and `preflight` reads it
with no GPU and no network. Run it first: the answer decides how many curves
Part 5 can draw.

A scorer that is missing is reported as missing. Nothing here substitutes one
scorer for another, and nothing fills a column with zeros.

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from design_pipeline import metrics, preflight
from design_pipeline.api import score_variants
from design_pipeline.errors import PipelineError
from design_pipeline.matrices import IndexMismatchError
from design_pipeline.structures import Complex, chain_sequence
from design_pipeline.variants import enumerate_variants, make_id

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

print(preflight.render(preflight.rows(MODELS)))
for note in preflight.notes(MODELS):
    print(f"\n{note}")

## Part 1 — the block, and the 156

The evidence CSV is one row per measurement, so the block has to be cut down to
the thing being calibrated: IgG variant rows, metric `KD`, a numeric value, and
a substitution list.

That last filter is not cosmetic. Three of the 159 numeric-KD rows are
`ablang_all_len9` designs whose CDR-H3 is nine residues instead of ten — a
deletion, not a substitution set, which is why their substitution field is empty
and their VH is 119 aa rather than 120. A `variant_id` in this package is a list
of substitutions and cannot express an indel, so those three are dropped here
and named, rather than quietly absorbed into a bin.

156 is what is left, and `CHINERY_EXPECTED_SIZES` is asserted against it.

The census printed first is the denominator the selection caveat is about: it is
the only place in this notebook where the 564 designs that produced no fit
appear at all.

In [ ]:
# The benchmark's own notation, kept separate from `variant_id`: the first is
# (H:/L:, mature-domain numbering), the second is this package's, over the
# chains 1N8Z actually has. Same three helpers as 03_adapt_calibration.ipynb.
TOKEN = re.compile(r"^(?P<chain>[A-Za-z0-9]+):(?P<wt>[A-Z])(?P<position>\d+)(?P<mut>[A-Z])$")


def parse_benchmark_variant(text):
    '''[(chain, position, wt, mut)] as the benchmark writes it.'''
    mutations = []
    for token in str(text).split(";"):
        match = TOKEN.match(token.strip())
        if match is None:
            raise ValueError(f"{text!r}: {token.strip()!r} is not a substitution like H:D31R")
        mutations.append((match["chain"], int(match["position"]), match["wt"], match["mut"]))
    if not mutations:
        raise ValueError(f"{text!r} names no substitution")
    return mutations


def remap(mutations):
    '''The same mutations over 1N8Z's chain ids. An unmapped chain is an error.'''
    out = []
    for chain, position, wt, mut in mutations:
        if chain not in CHAIN_MAP:
            raise KeyError(f"chain {chain!r} is not in CHAIN_MAP {CHAIN_MAP}")
        out.append((CHAIN_MAP[chain], position, wt, mut))
    return out


def label_of(mutations):
    return "+".join(f"{c}:{w}{p}{m}" for c, p, w, m in sorted(mutations))


raw = pd.read_csv(EVIDENCE, low_memory=False)
chinery_all = raw[(raw["study"] == CHINERY_STUDY) & (raw["format"] == CHINERY_FORMAT)]
submitted = chinery_all[chinery_all["record_type"] == "variant_outcome"]

census = pd.Series(
    {
        "numeric KD (a clean 1:1 fit)": int((submitted["metric"] == "KD").sum()),
        "N/A (no fit)": int((submitted["raw_KD_M"] == "N/A").sum()),
        "I.C. (binding seen, not reliably fit)": int((submitted["raw_KD_M"] == "I.C.").sum()),
        "blank": int(submitted["raw_KD_M"].isna().sum()),
    },
    name="designs",
)
print(f"block {CHINERY_STUDY!r} / {CHINERY_FORMAT!r}: {len(submitted)} designs submitted")
print(census.to_frame().to_string())
if int(census.sum()) != len(submitted):
    raise ValueError("the census does not account for every submitted design")
print(f"\n{census.iloc[0]} of {len(submitted)} returned a number. "
      f"Everything below is conditioned on that, and the CSV is explicit that "
      f"N/A and blank are NOT demonstrated non-binding.")

value = pd.to_numeric(chinery_all["value"], errors="coerce")
chinery_kd = chinery_all[
    (chinery_all["record_type"] == "variant_outcome")
    & (chinery_all["metric"] == "KD")
    & value.notna()
]
has_subs = chinery_kd["substitutions_mature_domain_1based"].notna() & (
    chinery_kd["substitutions_mature_domain_1based"].astype(str).str.strip() != ""
)
dropped = chinery_kd[~has_subs]
print(f"\n{len(chinery_kd)} numeric-KD variant rows; dropping {len(dropped)} with no "
      f"substitution list (a length change, not a substitution set):")
for row in dropped.itertuples():
    print(f"  {row.evidence_id}  {row.variant:<10} {row.design_method_or_control:<18}"
          f" VH {len(str(row.full_VH_sequence))} aa   KD {row.value} {row.units}")
chinery_kd = chinery_kd[has_subs]

controls = chinery_all[chinery_all["record_type"] == "assay_control"]
control_kd = pd.to_numeric(controls["value"], errors="coerce").dropna()
print(f"\n{len(control_kd)} wild-type control wells: KD {control_kd.min():.3f} to "
      f"{control_kd.max():.3f} nM ({control_kd.max() / control_kd.min():.1f}-fold across "
      f"plates), no matched parent assigned in the file")

## Part 2 — the numbering, checked three ways, before a model is loaded

The benchmark column is `substitutions_mature_domain_1based`: 1-based into the
mature VH. A `variant_id` here is 1-based into *the chain as given in the
structure*. Those agree only if the heavy chain of the PDB starts at VH residue
1 with no gaps and no insertion codes — and "only if" is not a thing to assume,
because a set numbered in Kabat or IMGT would land every substitution a few
residues off and still score perfectly cleanly. That is the failure mode worth
a cell.

**Check 1, the parent against the structure.** The block's wild-type VH must be
a prefix of 1N8Z chain `B`; if it is, the block's positions are this package's
positions. Which control well *is* the parent is decided by that agreement
rather than by a label, because `assay_control` in this block is a mixture —
only 8 of its 40 wells are trastuzumab, and the rest are benchmark controls
carried over from another study whose CDR-H3 is a different loop entirely. So
`controls[...].iloc[0]` is the parent today and is one re-sort of the CSV away
from not being, which is why this part asks the coordinates instead.

**Check 2, each design against itself.** Every design's `full_VH_sequence` must
be the parent VH with exactly its own substitutions applied, at the residues its
substitution string names.

**Check 3, free, from another route entirely.**
`trastuzumab_foldx_alanine_scan.csv` already scanned chain `B`, so the wild-type
residues at the ten redesigned positions can be read out of a file produced by
a different tool on a different day and compared. Its `ddg_bind` column is also
the useful prior on this loop: of the ten positions the block redesigns, W99 and
Y105 are the two whose side chains carry the binding.

Together the three tie the substitution strings, the deposited sequences and the
coordinates to the same molecule. Any of them failing is an error, not a
warning — a scorer run on a misaligned table produces numbers, and that is the
problem.

In [ ]:
CANDIDATES = (
    Path("data/structures/1N8Z.pdb"),
    Path("data/1N8Z.pdb"),
    Path.home() / ".cache" / "design_pipeline" / "pdb" / "1n8z.pdb",
)


def resolve_pdb():
    if PDB_PATH is not None:
        path = Path(PDB_PATH).expanduser()
        if not path.is_file():
            raise FileNotFoundError(f"PDB_PATH does not exist: {path}")
        return path
    for candidate in CANDIDATES:
        if candidate.is_file():
            return candidate
    looked = "\n".join(f"  {c}" for c in CANDIDATES)
    raise FileNotFoundError(f"no local 1N8Z found. Looked in:\n{looked}\n"
                            f"Set PDB_PATH to the copy you mean to use.")


pdb = resolve_pdb()
sequences = {c: chain_sequence(pdb, c) for c in (HEAVY_CHAIN, LIGHT_CHAIN, ANTIGEN_CHAIN)}
print(f"structure: {pdb}")
for chain, sequence in sequences.items():
    print(f"  chain {chain}: {len(sequence):>4} residues")

# --- check 1: the block's own wild type against the structure ---------------
# Which control is the parent is decided by the structure, not by a label and
# not by row order. `assay_control` in this block is a mixture: some wells are
# trastuzumab, and the rest are benchmark controls carried over from another
# study whose CDR-H3 is not the parent loop at all. Taking .iloc[0] would work
# by luck of ordering today and break silently the day the CSV is re-sorted.
control_vh = sorted({str(s) for s in controls["full_VH_sequence"].dropna()})
parents = [s for s in control_vh if sequences[HEAVY_CHAIN].startswith(s)]
if len(set(parents)) != 1:
    raise IndexMismatchError(
        f"{len(set(parents))} of the {len(control_vh)} distinct control VH sequences are a "
        f"prefix of 1N8Z chain {HEAVY_CHAIN}, and exactly one should be. Either "
        f"'mature-domain sequential' numbering is not 1-based into this chain - in which "
        f"case every position in this block would be read against the wrong residue - or "
        f"this is not the structure the block is about."
    )
wt_vh = parents[0]
wells = int((controls["full_VH_sequence"] == wt_vh).sum())
print(f"\n{len(control_vh)} distinct VH sequences across {len(controls)} control wells; "
      f"exactly one is a prefix of chain {HEAVY_CHAIN}")
print(f"the parent is that one ({len(wt_vh)} aa, {wells} wells), so the block's positions "
      f"are this package's positions")

# --- check 2: every design against its own supplied sequence ----------------
records, rebuilt_mismatch = [], []
for row in chinery_kd.itertuples():
    benchmark_mutations = parse_benchmark_variant(row.substitutions_mature_domain_1based)
    structure_mutations = remap(benchmark_mutations)
    residues = list(wt_vh)
    for _, position, wt, mut in benchmark_mutations:
        if residues[position - 1] != wt:
            rebuilt_mismatch.append((row.variant, f"{wt}{position}{mut}", residues[position - 1]))
            continue
        residues[position - 1] = mut
    if "".join(residues) != str(row.full_VH_sequence):
        rebuilt_mismatch.append((row.variant, "whole VH", "rebuilt sequence differs"))
    records.append({
        "evidence_id": row.evidence_id,
        "well": row.variant,
        "label": label_of(benchmark_mutations),
        "variant_id": make_id(structure_mutations),
        "n_mut": len(structure_mutations),
        "kd_nM": float(row.value),
        "design": row.design_method_or_control,
        "mutations": tuple(sorted(structure_mutations)),
    })

if rebuilt_mismatch:
    raise IndexMismatchError(
        f"{len(rebuilt_mismatch)} variant(s) whose substitution list does not rebuild "
        f"their own supplied VH sequence, e.g. "
        + "; ".join(f"{w}: {t} ({h})" for w, t, h in rebuilt_mismatch[:5])
        + ". The substitutions and the sequences in this block disagree."
    )

chinery = pd.DataFrame(records).sort_values(["n_mut", "kd_nM"]).reset_index(drop=True)
if chinery["variant_id"].duplicated().any():
    raise ValueError("a variant appears twice; grouping would have to average two KDs")
if len(chinery) != CHINERY_EXPECTED:
    raise ValueError(f"expected {CHINERY_EXPECTED} variants, grouped {len(chinery)}")
chinery_sizes = chinery["n_mut"].value_counts().to_dict()
if chinery_sizes != CHINERY_EXPECTED_SIZES:
    raise ValueError(f"expected {CHINERY_EXPECTED_SIZES} by substitution count, got {chinery_sizes}")
print(f"\nevery variant's supplied VH is the parent plus its own substitutions "
      f"({len(chinery)} of {len(chinery)})")

# --- check 3, free: the same wild type out of a file made by another route --
sites = sorted({(c, p) for row in chinery.itertuples() for c, p, _, _ in row.mutations})
scan_path = Path("trastuzumab_foldx_alanine_scan.csv")
if scan_path.is_file():
    scan = pd.read_csv(scan_path)
    at_sites = scan[
        (scan["chain"] == HEAVY_CHAIN) & scan["resseq"].isin([p for _, p in sites])
    ].sort_values("resseq")
    from_scan = "".join(at_sites["wt"])
    from_pdb = "".join(sequences[HEAVY_CHAIN][p - 1] for _, p in sites)
    if from_scan != from_pdb:
        raise IndexMismatchError(
            f"{scan_path} disagrees with the coordinates at the redesigned sites: "
            f"{from_scan} vs {from_pdb}"
        )
    print(f"the alanine scan agrees on the wild type at these {len(sites)} sites: {from_pdb}")
    print(at_sites[["chain", "resseq", "wt", "ddg_bind", "status"]].to_string(index=False))
    print("(that ddg_bind is the alanine scan's number, not this notebook's: it is the "
          "prior on which of the ten positions carry binding - W99 and Y105 do)")

print(f"\n{len(chinery)} variants, {sum(chinery['n_mut'])} substitutions over "
      f"{len(sites)} sites: {', '.join(f'{c}:{p}' for c, p in sites)}")
print("by substitution count: "
      + ", ".join(f"{n} with {k}" for k, n in sorted(chinery_sizes.items())))
print(f"KD {chinery['kd_nM'].min():.3f} to {chinery['kd_nM'].max():,.0f} nM "
      f"(median {chinery['kd_nM'].median():.1f}), lower is tighter")
print("designs: " + ", ".join(f"{k} {v}" for k, v in chinery["design"].value_counts().items()))

## Part 3 — one `variant_id` per design

`enumerate_variants(..., only=[...])` is the path the package already provides
for "score this measured set and nothing else". It is also the one place
multi-mutation ids are accepted, and it re-checks every wild-type residue
against the structure on the way in — so Part 2's first check gets made a third
time, by the package rather than by this notebook, which is the version that
counts.

The table is one row per substitution, with several rows sharing a
`variant_id`. That is the schema the scorers read, and it is why Part 5
collapses to one row per candidate before correlating anything: 156 designs are
730 rows, and correlating the rows would weight a six-substitution design six
times as heavily as a single.

Without ANARCI the `imgt` column comes back empty and warns. Nothing here joins
on it.

In [ ]:
complex_ = Complex(
    paths=(pdb,),
    output_dir=out_dir,
    config={"model_name": "local_file", "source": str(pdb)},
)

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="anarci is not installed")
    table = enumerate_variants(
        complex_,
        HEAVY_CHAIN,
        out_dir / "variants",
        only=chinery["variant_id"].tolist(),
        structure=pdb,
    )

print(f"{table.rows} rows, {table.candidates} candidates -> {table.path}")
if table.candidates != len(chinery):
    raise ValueError(f"{table.candidates} candidates for {len(chinery)} designs")
print(f"{table.rows} rows for {table.candidates} candidates: "
      f"{table.rows / table.candidates:.2f} substitutions per design on average\n")
print(table.frame.head(8).to_string(index=False))

## Part 4 — score, cheapest first

One `score_variants` call per model, because `**kwargs` go to *every* model
named in a call and these three want different ones: AntiFold needs chain ids,
FoldX reads its binary out of the environment. Each call joins its column into
the table the previous one returned, so this part ends with one table carrying
one `score_<model>` column per scorer that ran.

All three take a candidate's substitutions **together**, which is why only
these three are here:

| scorer | per candidate | stored sign |
|---|---|---|
| `antifold_seq` | one inverse-folding pass over the whole mutant chain; the change in total log-likelihood against wild type, in nats | already higher-is-better |
| `foldx` | `BuildModel` with every substitution on one `individual_list.txt` line, then `AnalyseComplex` on mutant and wild type, antibody against antigen; ddG of interaction | ddG negated |
| `gearbind` | a predicted ddG of binding | ddG negated |

`score_variants` normalizes at the adapter boundary: **`score_<model>` is
higher-is-better-binding** whatever the tool's own convention, and
`score_<model>_raw` keeps the unnormalized number so a convention we got
backwards is recoverable without re-running anything.

The plain `antifold` matrix scorer is deliberately **not** in this list. It
answers a single substitution from one forward pass and a per-position lookup,
so asking it about six means adding six independent numbers — which is the
assumption under test, not a way to test it. `antifold_seq` exists precisely
because it conditions on the whole mutant. (Part 6 of
`03_adapt_calibration.ipynb` does report the matrix lookup on the 14 singles,
where it is a fair question.)

Caching is per (structure, candidate set, model, kwargs), so a re-run is free
and adding a scorer does not invalidate one that already ran.

### Before running this part

* `antifold_seq` needs Env B: `DESIGN_PIPELINE_ANTIFOLD`, or `ANTIFOLD_PYTHON`
  above. About 0.7 s per candidate on a CPU, so roughly two minutes for 156.
* `foldx` needs `FOLDX_BINARY` and `DESIGN_PIPELINE_FOLDX_PDB`. It is
  `FOLDX_RUNS` BuildModel runs plus two AnalyseComplex calls per candidate:
  **hours, not minutes.** Point `DESIGN_PIPELINE_FOLDX_PDB` at the trimmed,
  repaired structure the alanine scan used, or it will be days.
* `gearbind` needs an image you built. `GEARBIND_IMAGE` wires it in; left None,
  it is reported as unavailable and the notebook carries on.

In [ ]:
scored = table
ran, skipped = [], {}


def run_model(current, model, **kwargs):
    """One scorer onto the table, or a named skip. Returns the table either way.

    A scorer that cannot run here is recorded in `skipped` with the reason the
    package gave and the notebook carries on: a missing FoldX licence should
    cost you the FoldX curve, not the other two.
    """
    try:
        result = score_variants(
            complex_,
            current,
            out_dir / "scores",
            models=[model],
            structure=pdb,
            max_variants=MAX_VARIANTS,
            cache=CACHE,
            **{k: v for k, v in kwargs.items() if v is not None},
        )
    except PipelineError as exc:
        skipped[model] = f"{type(exc).__name__}: {exc}"
        print(f"{model}: SKIPPED\n  {type(exc).__name__}: {exc}\n")
        return current
    column = f"score_{model}"
    one_per_candidate = result.frame.drop_duplicates("variant_id")
    answered = int(one_per_candidate[column].notna().sum())
    print(f"{model}: {answered} of {result.candidates} candidates scored")
    print(f"  -> {result.path}")
    if answered == 0:
        skipped[model] = "ran, but scored nothing"
        print(f"  every value is empty, so {model} is treated as skipped\n")
        return result
    ran.append(model)
    return result


scored = run_model(
    scored,
    "antifold_seq",
    executable=ANTIFOLD_PYTHON,
    heavy=HEAVY_CHAIN,
    light=LIGHT_CHAIN,
    device=ANTIFOLD_DEVICE,
    weights=ANTIFOLD_WEIGHTS,
)

In [ ]:
# FoldX is configured entirely by environment, not by flags: _foldx_score.py
# takes --pdb, --out_dir and --mutations_csv and nothing else, so a `runs=` or
# `workers=` keyword would reach argparse and fail the whole call. These are set
# here instead, and only when the parameters cell asked for them.
import os

for name, setting in (("FOLDX_RUNS", FOLDX_RUNS), ("FOLDX_WORKERS", FOLDX_WORKERS)):
    if setting is not None:
        os.environ[name] = str(setting)

FOLDX_ENV = (
    ("FOLDX_BINARY", "required - the call will fail without it"),
    ("DESIGN_PIPELINE_FOLDX_PDB", "falls back to --pdb, i.e. the whole of 1N8Z"),
    ("DESIGN_PIPELINE_FOLDX", "falls back to python3 on PATH"),
    ("FOLDX_GROUPS", "defaults to AB,C, already antibody-vs-antigen here"),
    ("FOLDX_RUNS", "defaults to 3"),
    ("FOLDX_WORKERS", "defaults to 4"),
)
for name, fallback in FOLDX_ENV:
    value = os.environ.get(name)
    print(f"  {name:<26} {value if value else f'unset ({fallback})'}")

# Heads-up on the cache: the key is (structure, candidate set, model, kwargs),
# and these are environment, not kwargs. Changing FOLDX_RUNS will NOT invalidate
# a cached run - pass cache=False, or delete the run directory, if you mean to
# re-run with different settings.
#
# Hours, not minutes. FOLDX_WORKERS is what makes it bearable and
# DESIGN_PIPELINE_FOLDX_PDB is what makes it possible: on the full 1N8Z, with
# HER2 domain IV at 600-odd residues, BuildModel per candidate is days.
scored = run_model(scored, "foldx", executable=FOLDX_PYTHON)

In [ ]:
# GearBind ships no image, so the registry carries tool=None and `run` raises
# out of routes. Wiring an image is the caller's job, and this is the one line
# that does it; with GEARBIND_IMAGE left None nothing is registered and the
# skip is reported with the route from routes.py.
if GEARBIND_IMAGE is not None:
    from dataclasses import replace

    from design_pipeline import REGISTRY
    from design_pipeline.adapters import _scorers

    REGISTRY["gearbind"] = replace(
        _scorers.gearbind, tool=_scorers.matrix_container(GEARBIND_IMAGE)
    )
    print(f"gearbind -> {GEARBIND_IMAGE}")

scored = run_model(scored, "gearbind")

print(f"\nscorers that ran: {', '.join(ran) if ran else 'none'}")
for model, reason in skipped.items():
    print(f"skipped {model}: {reason.splitlines()[0]}")
if not ran:
    raise RuntimeError(
        "no scorer ran, so there is no curve to draw. Part 0's table says what "
        "each one needs; the cheapest to satisfy is antifold_seq."
    )

## Part 5 — Spearman within each substitution count

The scored table is joined back onto the block, one row per design.

**The sign.** Lower KD is tighter binding and `score_<model>` is
higher-is-better, so a scorer that agrees with the measurement correlates
**positively with −KD**. Spearman is rank-based, so −KD, −log KD and pKD give
one and the same number; the raw-KD figure the tables also print is simply this
one negated. The curve is plotted against −KD, so that **up is a scorer getting
it right and the zero line is no skill at all.**

Bins with fewer than `MIN_BIN` designs are reported but drawn hollow, and
anything under `metrics.MIN_PAIRS` is not computed at all — at n = 4 a rank
correlation is noise with a number attached, and `metrics.spearman` returns NaN
rather than 0.0 so that the absence of a claim does not read as a claim of no
skill.

In [ ]:
# --- join the scores back onto the block, one row per design ----------------
one_per_candidate = scored.frame.drop_duplicates("variant_id")
columns = ["variant_id"] + [f"score_{m}" for m in ran] + [f"score_{m}_raw" for m in ran]
columns = [c for c in columns if c in one_per_candidate.columns]
panel = chinery.merge(one_per_candidate[columns], on="variant_id", how="left")
if len(panel) != len(chinery):
    raise ValueError(f"the join changed the row count: {len(chinery)} -> {len(panel)}")

# The fit rate per substitution count: of everything submitted at this size,
# how much of it produced a number at all. This is the selection, quantified.
# pd.isna, not `or ""`: a NaN is truthy, so `value or ""` would turn a design
# with no substitution list into the string "nan" and count it as a single.
submitted_sizes = {}
for value in submitted["substitutions_mature_domain_1based"]:
    text = "" if pd.isna(value) else str(value).strip()
    size = len([t for t in text.split(";") if t.strip()])
    submitted_sizes[size] = submitted_sizes.get(size, 0) + 1
if sum(submitted_sizes.values()) != len(submitted):
    raise ValueError("the per-size census does not account for every submitted design")

target = -panel["kd_nM"].to_numpy()


def spearman_of(column, mask=None):
    """(rho, n) of a score column against -KD, so positive means the scorer agrees."""
    rows = panel if mask is None else panel[mask]
    values = rows[column].to_numpy(dtype=float)
    return metrics.spearman(values, -rows["kd_nM"].to_numpy()), int(np.isfinite(values).sum())


def cell(rho):
    return "-" if rho != rho else f"{rho:+.3f}"


# --- the within-size table, which the curve in the next cell plots ----------
head = f"{'n_sub':>6}{'designs':>9}{'submitted':>11}{'fit %':>8}"
head += "".join(f"{m:>15}" for m in ran)
head += f"{'median KD nM':>14}{'KD span':>11}"
print(f"Spearman against -KD within a fixed substitution count "
      f"(positive = the scorer agrees; n < {metrics.MIN_PAIRS} is not computed)")
print(head)
print("-" * len(head))

curve = []
for size in sorted(panel["n_mut"].unique()):
    at_size = panel["n_mut"] == size
    rows = panel[at_size]
    total = submitted_sizes.get(size, len(rows))
    record = {
        "n_sub": int(size),
        "designs": int(len(rows)),
        "submitted": int(total),
        "fit_pct": 100.0 * len(rows) / total,
        "median_kd_nM": float(rows["kd_nM"].median()),
        "kd_span": float(rows["kd_nM"].max() / rows["kd_nM"].min()),
    }
    line = (f"{size:>6}{len(rows):>9}{total:>11}{record['fit_pct']:>7.1f}%")
    for model in ran:
        rho, n = spearman_of(f"score_{model}", at_size)
        record[model] = rho
        record[f"{model}_n"] = n
        line += f"{cell(rho):>15}"
    line += f"{record['median_kd_nM']:>14.1f}{record['kd_span']:>10,.0f}x"
    print(line)
    curve.append(record)

curve = pd.DataFrame(curve).set_index("n_sub")
print("-" * len(head))
print(f"bins under {MIN_BIN} designs are drawn hollow in the plot and are not evidence "
      f"of much on their own.")
print("'fit %' is the selection: of everything submitted at this size, how much "
      "returned a number.")

### What the 2026-10-31 run recorded, for comparison

The table above is computed by this notebook and is blank until it has been run.
What follows is **not this notebook's output.** It is what Part 6 of
`03_adapt_calibration.ipynb` printed on 2026-10-31, over this same 156
candidates and the same structure, quoted here so that the check in the next
cell has something visible to check against and so this notebook is readable
before it is run.

It is a quotation rather than a transplant because the code that produced it is
not the code above: it reported a matrix-lookup column, had no fit-rate columns
and no FoldX column, and ran two scorers rather than up to three. Pasting it
into a cell here would attach output to code that cannot produce it.

```text
scorer                          n   Spearman vs -KD   vs KD as reported
-----------------------------------------------------------------------
antifold_seq (all)            156            +0.237              -0.237
antifold matrix (singles)      14            +0.147              -0.147
mutation count alone          156            +0.461              -0.461
-----------------------------------------------------------------------

within a fixed mutation count, where counting cannot help (n < 3 is not computed):
 n_mut  variants  antifold_seq    matrix  median KD nM     KD span
------------------------------------------------------------------
     1        14        +0.415    +0.147           4.9        974x
     2        13        +0.346         -          28.0       5437x
     3        17        +0.145         -          19.9      18730x
     4        32        -0.011         -          81.5       3483x
     5        27        -0.277         -         138.0       1075x
     6        23        -0.275         -         127.5       4789x
     7        14        -0.277         -         159.5         90x
     8         8        -0.405         -         284.0        111x
     9         4        +0.000         -         190.7          5x
    10         4        -0.400         -         203.0          3x
------------------------------------------------------------------

antifold_seq pooled: +0.237; weighted mean of the within-size figures: -0.065
over 156 of 156 variants, in 10 size groups.
```

Three things to read out of it before running anything:

* **The within-size figures cross zero at four substitutions and stay there.**
  The `antifold_seq` column is the curve this notebook plots, so its shape is
  already visible above: +0.415 at one substitution, +0.145 at three, and
  negative from four to ten bar one tied bin.
* **Pooled, `antifold_seq` scores +0.237 against substitution count's +0.461.**
  It loses to counting. The weighted mean of the within-size figures is −0.065,
  so very nearly all of that pooled correlation was the size effect.
* **Median KD climbs with size — 4.9 nM at one substitution to about 200 nM at
  nine or ten — but not monotonically:** it falls at three, at six and at nine.
  That is the confound sitting in the raw measurement before any scorer is
  involved, and the non-monotonicity is a reminder that these bins are small.

The same source cell also printed the ten tightest binders with their rank by
`antifold_seq`; a cut at its top ten would have caught one of them. This
notebook does not reproduce that table — it is in `03_adapt_calibration.ipynb`
if you want it, along with the matrix-lookup column, which is deliberately not
recomputed here because summing independent single-mutation lookups is the
assumption under test.

In [ ]:
# --- the regression check against the figures already on record ------------
# Part 6 of 03_adapt_calibration.ipynb ran antifold_seq over this same 156 on
# 2026-10-31. Those per-size numbers are quoted in full in the cell above and
# pinned as KNOWN_ANTIFOLD_SEQ in the parameters cell. If this run disagrees,
# something in the pipeline moved between then and now, and that is worth
# stopping for rather than noticing six months later.
if "antifold_seq" in ran:
    drift = []
    print(f"{'n_sub':>6}{'recorded':>11}{'here':>9}{'delta':>9}")
    print("-" * 35)
    for size, expected in sorted(KNOWN_ANTIFOLD_SEQ.items()):
        rho = curve["antifold_seq"].get(size, float("nan"))
        delta = rho - expected
        flag = "" if abs(delta) <= KNOWN_TOLERANCE else "   <- moved"
        print(f"{size:>6}{expected:>+11.3f}{cell(rho):>9}{delta:>+9.3f}{flag}")
        if not (abs(delta) <= KNOWN_TOLERANCE):
            drift.append((size, expected, rho))
    print("-" * 35)
    if drift:
        raise ValueError(
            f"antifold_seq disagrees with the 2026-10-31 run at "
            f"{len(drift)} substitution count(s): "
            + "; ".join(f"n={s} recorded {e:+.3f}, got {r:+.3f}" for s, e, r in drift)
            + ". Either the scorer, the structure or the candidate set changed. "
            "Find out which before reading the curve."
        )
    print(f"antifold_seq reproduces the 2026-10-31 run to within "
          f"{KNOWN_TOLERANCE} at every substitution count.")
else:
    print("antifold_seq did not run, so there is nothing to check against the "
          "2026-10-31 figures.")

In [ ]:
# --- the deliverable: Spearman against substitution count -------------------
try:
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
except ImportError:
    plt = None
    print("matplotlib is not installed, so the curve is skipped. The table above "
          "is the same numbers; `pip install matplotlib` to draw it.")

if plt is not None:
    pooled = {m: spearman_of(f"score_{m}")[0] for m in ran}
    count_rho = metrics.spearman(-panel["n_mut"].to_numpy(dtype=float), target)
    sizes = curve.index.to_numpy()
    colours = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    figure, (top, bottom) = plt.subplots(
        2, 1, figsize=(9.5, 8.2), sharex=True,
        gridspec_kw={"height_ratios": [2.1, 1.0], "hspace": 0.12},
    )

    top.axhline(0.0, color="0.35", linewidth=1.0, zorder=1)
    for index, model in enumerate(ran):
        colour = colours[index % len(colours)]
        values = curve[model].to_numpy(dtype=float)
        big = curve["designs"].to_numpy() >= MIN_BIN
        top.plot(sizes, values, "-", color=colour, linewidth=1.8, zorder=3, label=model)
        top.plot(sizes[big], values[big], "o", color=colour, markersize=7, zorder=4)
        top.plot(sizes[~big], values[~big], "o", color=colour, markersize=7,
                 markerfacecolor="white", markeredgewidth=1.8, zorder=4)
        top.axhline(pooled[model], color=colour, linewidth=1.0, linestyle=":", zorder=2)

    top.axhline(count_rho, color="0.25", linewidth=1.3, linestyle="--", zorder=2)
    top.annotate(f"substitution count alone, pooled: {count_rho:+.3f}",
                 xy=(sizes[-1], count_rho), xytext=(-4, 5), textcoords="offset points",
                 ha="right", fontsize=8.5, color="0.25")
    top.set_ylabel("Spearman, score vs −KD\n(positive = the scorer agrees)")
    top.set_title("Does scorer accuracy hold as candidates get further from the parent?\n"
                  "Chinery 2026 CDR-H3 IgG panel, 156 designs with a 1:1 fit",
                  fontsize=11.5, loc="left")
    top.set_ylim(-1.0, 1.0)
    top.grid(axis="y", alpha=0.25)

    handles = [Line2D([], [], color=colours[i % len(colours)], marker="o", linewidth=1.8,
                      label=m) for i, m in enumerate(ran)]
    handles += [
        Line2D([], [], color="0.5", linestyle=":", label="that scorer, pooled over all 156"),
        Line2D([], [], color="0.25", linestyle="--", label="substitution count alone"),
        Line2D([], [], color="0.5", marker="o", linestyle="none", markerfacecolor="white",
               markeredgewidth=1.8, label=f"bin under {MIN_BIN} designs"),
    ]
    top.legend(handles=handles, fontsize=8.5, loc="lower left", framealpha=0.9)

    # The confound, on the same x axis, so it cannot be read separately from the
    # curve it could be causing.
    width = 0.38
    bottom.bar(sizes - width / 2, curve["fit_pct"], width, color="0.55",
               label="returned a numeric KD (% of submitted)")
    kd_axis = bottom.twinx()
    kd_axis.plot(sizes, curve["median_kd_nM"], "s-", color="firebrick", markersize=5,
                 linewidth=1.5, label="median KD of the survivors")
    kd_axis.set_yscale("log")
    kd_axis.set_ylabel("median KD, nM (log)", color="firebrick")
    kd_axis.tick_params(axis="y", colors="firebrick")
    for size, value, n in zip(sizes, curve["fit_pct"], curve["designs"]):
        bottom.annotate(f"n={n}", xy=(size - width / 2, value), xytext=(0, 3),
                        textcoords="offset points", ha="center", fontsize=8, color="0.3")
    bottom.set_ylabel("fit rate, %")
    bottom.set_xlabel("substitutions from the parent")
    bottom.set_xticks(sizes)
    bottom.set_ylim(0, 112)
    bottom.grid(axis="y", alpha=0.25)
    bottom.legend(fontsize=8.5, loc="upper right")
    kd_axis.legend(fontsize=8.5, loc="center right")

    figure.text(0.012, 0.012,
                "The 156 are the designs that produced a clean 1:1 fit out of 720 submitted, so they are selected on "
                "outcome:\nthe KD range is truncated and every correlation here is attenuated. The fit rate below shows "
                "that the selection is\nstrongly size-dependent, so a falling curve is also consistent with the large-size "
                "bins simply having been filtered harder.",
                fontsize=8, color="0.3", va="bottom")
    figure.subplots_adjust(bottom=0.155, top=0.9, left=0.1, right=0.9)

    figure.savefig(out_dir / "spearman_by_substitution_count.png", dpi=150)
    print(f"-> {out_dir / 'spearman_by_substitution_count.png'}")
    plt.show()

## Part 6 — the confounds, out loud

Three numbers belong next to the curve, because without them a pooled
correlation can be read as scorer skill when it is nothing of the kind.

**The pooled correlation.** One number over all 156, ignoring size. Reported,
but it is the number this notebook exists to distrust.

**Substitution count alone.** Spearman of `−n_sub` against `−KD`: a "scorer"
that knows nothing except that more change is worse. This is the baseline every
real scorer has to beat. A pooled figure at or below it is a correlation with
distance from the parent wearing a scorer's name, and the within-size curve is
the only way to tell the two apart — which is exactly why the curve, not the
pooled number, is the deliverable.

**The weighted mean of the within-size figures.** What is left of the pooled
correlation once counting cannot help. The gap between this and the pooled
number is the size effect, measured.

And the selection, restated with its own number: the fit rate against
substitution count. If that correlation is strongly negative, the large-size
bins are the survivors of a much harsher filter than the small-size ones, and
the curve's right-hand end is measuring a different population from its left.

In [ ]:
count_rho = metrics.spearman(-panel["n_mut"].to_numpy(dtype=float), target)
weights = curve["designs"].to_numpy(dtype=float)

print(f"{'scorer':<26}{'n':>5}{'vs -KD':>11}{'vs raw KD':>12}"
      f"{'within-size mean':>19}")
print("-" * 73)
for model in ran:
    rho, n = spearman_of(f"score_{model}")
    values = panel[f"score_{model}"].to_numpy(dtype=float)
    as_reported = metrics.spearman(values, panel["kd_nM"].to_numpy())
    within = curve[model].to_numpy(dtype=float)
    usable = np.isfinite(within)
    weighted = (
        float(np.sum(within[usable] * weights[usable]) / np.sum(weights[usable]))
        if usable.any() else float("nan")
    )
    print(f"{model:<26}{n:>5}{rho:>+11.3f}{as_reported:>+12.3f}{weighted:>+19.3f}")
print(f"{'substitution count alone':<26}{len(panel):>5}{count_rho:>+11.3f}"
      f"{metrics.spearman(-panel['n_mut'].to_numpy(dtype=float), panel['kd_nM'].to_numpy()):>+12.3f}"
      f"{'n/a':>19}")
print("-" * 73)
print("'vs raw KD' is the same rank correlation negated, since lower KD is tighter.")
print("'within-size mean' weights each substitution count by how many designs it has.")

print()
for model in ran:
    rho, _ = spearman_of(f"score_{model}")
    if rho != rho:
        continue
    if rho <= count_rho:
        print(f"{model}: pooled {rho:+.3f} does not beat substitution count alone "
              f"({count_rho:+.3f}). Pooled, it is a correlation with distance from "
              f"the parent.")
    else:
        print(f"{model}: pooled {rho:+.3f} beats substitution count alone "
              f"({count_rho:+.3f}) by {rho - count_rho:+.3f}.")

# The selection, as a number on the same footing as the rest.
fit_rho = metrics.spearman(
    curve.index.to_numpy(dtype=float), curve["fit_pct"].to_numpy(dtype=float)
)
print(f"\nfit rate against substitution count: {fit_rho:+.3f} over "
      f"{len(curve)} sizes "
      f"({curve['fit_pct'].iloc[0]:.0f}% at {curve.index[0]} substitution(s) down to "
      f"{curve['fit_pct'].iloc[-1]:.0f}% at {curve.index[-1]})")
print(f"{int(curve['designs'].sum())} of {int(curve['submitted'].sum())} designs with a "
      f"substitution list returned a number.")
print("So the right-hand bins are the survivors of a much harsher filter than the "
      "left-hand ones.\nA curve that falls with size is consistent with scorers "
      "degrading over distance AND with\nsurvivorship alone; this panel cannot "
      "separate them, and the unmeasured designs are the\nmissing evidence that "
      "would.")

## Part 7 — ADAPT, kept apart

The ADAPT block is the sizes-1-to-3 anchor: 22 variants with a measured
affinity fold-improvement, which is the calibration Track A has been leaning
on. It is run here on the same scorers so the anchor and the curve come out of
one pipeline — and then it is **left in its own table.**

It is not pooled with Part 5, in either direction, for three independent
reasons, any one of which would be enough:

* **Different molecule.** ADAPT is Fab, monovalent. Chinery is IgG captured
  through Fc on Carterra, two Fab arms against a surface-presented antigen. A
  design-dependent avidity term would survive pooling as a spurious signal.
* **Different measurement.** ADAPT reports fold-improvement against a
  batch-matched parent. Chinery has no matched wild-type fold at all, so its
  ground truth is a raw KD. One is a ratio against a control run alongside;
  the other is an absolute number from a plate with no such control.
* **Different sign.** Fold-improvement is higher-is-better, KD is
  lower-is-better. The correlations below are against fold-improvement
  **directly**, so positive still means the scorer agrees — but the two tables
  reach that convention by opposite routes, which is precisely the kind of
  thing a careless `pd.concat` gets wrong.

So the only comparison the two tables support is a qualitative one: at sizes 1
to 3, do the signs agree. Not the magnitudes, and not a joint correlation.

In [ ]:
if not SCORE_ADAPT:
    print("SCORE_ADAPT is False, so the ADAPT anchor is skipped.")
else:
    adapt_rows = raw[
        (raw["study"] == ADAPT_STUDY) & (raw["metric"] == "affinity_fold_improvement")
    ]
    adapt_records = []
    for row in adapt_rows.itertuples():
        benchmark_mutations = parse_benchmark_variant(row.variant)
        structure_mutations = remap(benchmark_mutations)
        adapt_records.append({
            "evidence_id": row.evidence_id,
            "label": label_of(benchmark_mutations),
            "variant_id": make_id(structure_mutations),
            "n_mut": len(structure_mutations),
            "fold": float(row.value),
            "format": row.format,
        })
    adapt = pd.DataFrame(adapt_records).sort_values(["n_mut", "fold"]).reset_index(drop=True)
    print(f"{ADAPT_STUDY}: {len(adapt)} variants, "
          f"format {sorted(set(adapt['format']))}, metric affinity_fold_improvement "
          f"(higher is better)")
    print("by substitution count: "
          + ", ".join(f"{n} with {k}" for k, n in sorted(adapt['n_mut'].value_counts().items())))
    print(f"fold-improvement {adapt['fold'].min():.1f} to {adapt['fold'].max():.1f}\n")

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="anarci is not installed")
        adapt_table = enumerate_variants(
            complex_, HEAVY_CHAIN, out_dir / "adapt_variants",
            only=adapt["variant_id"].tolist(), structure=pdb,
        )
    print(f"{adapt_table.rows} rows, {adapt_table.candidates} candidates\n")

    adapt_scored, adapt_ran = adapt_table, []
    for model in ran:
        kwargs = {}
        if model == "antifold_seq":
            kwargs = {"executable": ANTIFOLD_PYTHON, "heavy": HEAVY_CHAIN,
                      "light": LIGHT_CHAIN, "device": ANTIFOLD_DEVICE,
                      "weights": ANTIFOLD_WEIGHTS}
        elif model == "foldx":
            kwargs = {"executable": FOLDX_PYTHON}
        try:
            adapt_scored = score_variants(
                complex_, adapt_scored, out_dir / "adapt_scores", models=[model],
                structure=pdb, max_variants=MAX_VARIANTS, cache=CACHE,
                **{k: v for k, v in kwargs.items() if v is not None},
            )
            adapt_ran.append(model)
            print(f"{model}: scored the ADAPT anchor")
        except PipelineError as exc:
            print(f"{model}: skipped on ADAPT - {type(exc).__name__}: {exc}")

    adapt_panel = adapt.merge(
        adapt_scored.frame.drop_duplicates("variant_id")[
            ["variant_id"] + [f"score_{m}" for m in adapt_ran]
        ],
        on="variant_id", how="left",
    )

    print(f"\nADAPT: Spearman against fold-improvement directly "
          f"(positive = the scorer agrees). Fab, monovalent, batch-matched parent.")
    print("NOT comparable in magnitude with Part 5, and never pooled with it.")
    head = f"{'n_sub':>6}{'variants':>10}" + "".join(f"{m:>15}" for m in adapt_ran)
    print(head)
    print("-" * len(head))
    for size in sorted(adapt_panel["n_mut"].unique()):
        rows = adapt_panel[adapt_panel["n_mut"] == size]
        line = f"{size:>6}{len(rows):>10}"
        for model in adapt_ran:
            rho = metrics.spearman(
                rows[f"score_{model}"].to_numpy(dtype=float),
                rows["fold"].to_numpy(dtype=float),
            )
            line += f"{cell(rho):>15}"
        print(line)
    line = f"{'all':>6}{len(adapt_panel):>10}"
    for model in adapt_ran:
        rho = metrics.spearman(
            adapt_panel[f"score_{model}"].to_numpy(dtype=float),
            adapt_panel["fold"].to_numpy(dtype=float),
        )
        line += f"{cell(rho):>15}"
    print("-" * len(head))
    print(line)
    print("-" * len(head))
    print(f"(n < {metrics.MIN_PAIRS} is not computed, so the size-3 row with 4 "
          f"variants is at the edge of meaning anything.)")

## What the curve licenses, and what it does not

Fill this in from the run, not from the expectation. Two things to write down
while the output is on screen:

**Where each scorer stops being usable.** Read the substitution count at which
its within-size correlation crosses zero and stays there. That number, not the
pooled figure, is the one to carry into the Track A filtering docs, because it
is the honest answer to "how far from the parent can we still rank".

**Whether the fall is skill or survivorship.** Part 6 prints the fit rate
against substitution count. If that is strongly negative — and on this block it
is — then the large-size bins are the survivors of a much harder filter, and the
curve's decline has two candidate explanations that this data cannot separate.
Write down both. A curve presented as "scorers degrade past three substitutions"
without the fit-rate panel beside it is an overclaim.

### What would actually settle it

The unmeasured designs are the missing evidence, and nothing in this notebook
can recover them. Three things would:

* **A panel with a batch-matched parent on every plate,** so the measurement is
  a fold-improvement and not an absolute KD read against controls from
  elsewhere.
* **Fab, or a monovalent format,** so the avidity term is gone and the
  calibration transfers to what the scorers are actually modelling — a single
  interface.
* **A design set that is not filtered on whether it bound.** Even a coarse
  three-bin readout (tight / measurable / nothing) over all 720 would be worth
  more here than 156 clean KDs, because it is the 564 that carry the
  size-dependence.

Until one of those exists, the defensible claim from this notebook is the
narrow one: **on the only multi-substitution panel available, no scorer in this
repo ranks a design five or more substitutions from the parent better than
chance, and the pooled correlation that appears to show otherwise is mostly a
correlation with substitution count.** That is enough to stop trusting a pooled
number. It is not enough to put a number on how much accuracy is lost per
substitution.